# Lecture 4：从 token 到位置表示
按顺序运行。使用公开 Qwen3 tokenizer；小词表实验无需下载模型权重。

In [2]:
import os
os.environ["USE_TF"] = "0"
from pathlib import Path
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoTokenizer
print(torch.__version__)
torch.manual_seed(42)

2.8.0


## 实验 1：中英文分词与批量输入
可通过 LECTURE4_TOKENIZER_PATH 指定本地 tokenizer。默认使用公开模型或课程缓存。

In [3]:
cache = Path.home() / ".cache/llm-course/qwen3-tokenizer"
source = os.environ.get("LECTURE4_TOKENIZER_PATH") or (str(cache) if (cache / "tokenizer.json").exists() else "Qwen/Qwen3-0.6B")
tokenizer = AutoTokenizer.from_pretrained(source)
for text in ["Nanjing University", "南京大学", "LLM中的embedding"]:
    ids = tokenizer.encode(text, add_special_tokens=False)
    print(text, tokenizer.tokenize(text), ids, len(ids))
    assert tokenizer.decode(ids) == text
if tokenizer.pad_token_id is None:
    if tokenizer.eos_token_id is None:
        raise ValueError("请配置 tokenizer 的 pad token")
    tokenizer.pad_token = tokenizer.eos_token
batch = tokenizer(["你好", "南京大学"], padding=True, return_tensors="pt")
print("token IDs:\n", batch["input_ids"])
print("attention mask:\n", batch["attention_mask"])
print("每句话的有效 token 数:", batch["attention_mask"].sum(dim=1).tolist())
assert batch["input_ids"].shape == batch["attention_mask"].shape

Nanjing University ['N', 'anj', 'ing', 'ĠUniversity'] [45, 52091, 287, 3822] 4
南京大学 ['åįĹäº¬', 'å¤§åŃ¦'] [102034, 99562] 2
LLM中的embedding ['LL', 'M', 'ä¸ŃçļĦ', 'embedding'] [4086, 44, 101047, 94611] 4
token IDs:
 tensor([[108386, 151643],
        [102034,  99562]])
attention mask:
 tensor([[1, 0],
        [1, 1]])
每句话的有效 token 数: [1, 2]


## 实验 2：小词表查表
用一个 4 × 4 的矩阵观察 token ID 如何选择行，以及输出形状如何变化。

In [4]:
I = torch.tensor([[0, 3, 3], [2, 1, 0]])
T = torch.arange(16, dtype=torch.float32).reshape(4, 4)
E = T[I]
print("词表：\n", T)
print("token IDs：\n", I)
print("查表结果：\n", E)
assert E.shape == (2, 3, 4)
assert torch.equal(E[0, 1], E[0, 2])


词表：
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]])
token IDs：
 tensor([[0, 3, 3],
        [2, 1, 0]])
查表结果：
 tensor([[[ 0.,  1.,  2.,  3.],
         [12., 13., 14., 15.],
         [12., 13., 14., 15.]],

        [[ 8.,  9., 10., 11.],
         [ 4.,  5.,  6.,  7.],
         [ 0.,  1.,  2.,  3.]]])


## 实验 2.1：真实 Qwen3-0.6B 的输入 Embedding
对应讲义「从小词表到真实模型」。先预测输出形状，再运行。
只读取本地模型，不联网；CPU float32 加载约需数 GB 内存。
这里观察的是输入表示，不是文本生成，也尚未应用 RoPE。


In [5]:
os.environ["USE_TF"] = "0"
from transformers import AutoModelForCausalLM

model_path = Path(os.environ.get("LECTURE4_MODEL_PATH", str(cache))).expanduser()
if not (model_path / "model.safetensors").exists():
    raise FileNotFoundError(f"请将 LECTURE4_MODEL_PATH 指向完整的 Qwen3-0.6B 目录：{model_path}")
qwen_tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
qwen = AutoModelForCausalLM.from_pretrained(
    model_path, local_files_only=True, dtype=torch.float32
).eval()
input_embedding = qwen.get_input_embeddings()
print("模型：", qwen.config.model_type)
print("词表形状 [v, e]：", tuple(input_embedding.weight.shape))
print("层数：", qwen.config.num_hidden_layers)

print(qwen)

模型： qwen3
词表形状 [v, e]： (151936, 1024)
层数： 28
Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1

### 从文本到 [batch, sequence, hidden]
观察中文、英文的实际 token 数。对每个 token 只展示向量的前 6 维。
token 字符串是内部表示；单个 token 未必能独立解码成完整汉字。


In [5]:
for sentence in ["南京大学", "Nanjing University"]:
    encoded = qwen_tokenizer(sentence, add_special_tokens=False, return_tensors="pt")
    token_ids = encoded["input_ids"]
    with torch.no_grad():
        vectors = input_embedding(token_ids)
    print("\n文本：", sentence)
    print("IDs：", token_ids.tolist(), "输出形状：", tuple(vectors.shape))
    assert vectors.shape == (*token_ids.shape, qwen.config.hidden_size)
    for token, token_id, vector in zip(qwen_tokenizer.convert_ids_to_tokens(token_ids[0].tolist()), token_ids[0], vectors[0]):
        print(repr(token), token_id.item(), vector[:6].tolist())
    torch.testing.assert_close(vectors, input_embedding.weight[token_ids])
print("真实模型的 Embedding 与按行查表完全一致")



文本： 南京大学
IDs： [[102034, 99562]] 输出形状： (1, 2, 1024)
'åįĹäº¬' 102034 [0.0155029296875, -0.056640625, 0.02392578125, 0.052001953125, -0.026123046875, -0.036865234375]
'å¤§åŃ¦' 99562 [0.039794921875, -0.051513671875, -0.0172119140625, 0.05078125, 0.0079345703125, -0.056884765625]

文本： Nanjing University
IDs： [[45, 52091, 287, 3822]] 输出形状： (1, 4, 1024)
'N' 45 [-0.0341796875, 0.0281982421875, -0.046630859375, -0.006317138671875, -0.0159912109375, 0.0045166015625]
'anj' 52091 [0.0074462890625, -0.0238037109375, 0.0296630859375, 0.04443359375, 0.01556396484375, -0.0177001953125]
'ing' 287 [0.05078125, 0.03759765625, -0.05615234375, -0.0263671875, -0.04736328125, -0.005218505859375]
'ĠUniversity' 3822 [0.0260009765625, 0.0174560546875, -0.01123046875, 0.046630859375, 0.0235595703125, -0.031494140625]
真实模型的 Embedding 与按行查表完全一致


### 同一 token 的输入向量会随上下文变化吗？
先重复同一 ID，验证输入 embedding 相同；之后 Transformer 才把上下文和位置信息融入表示。
补齐 token 也会查表得到向量，attention mask 负责后续注意力屏蔽，并不自动把 embedding 清零。


In [6]:
probe_id = qwen_tokenizer.encode("大学", add_special_tokens=False)[0]
repeated_ids = torch.tensor([[probe_id, probe_id]])
with torch.no_grad():
    repeated_vectors = input_embedding(repeated_ids)
torch.testing.assert_close(repeated_vectors[:, 0], repeated_vectors[:, 1])
if qwen_tokenizer.pad_token_id is None:
    qwen_tokenizer.pad_token = qwen_tokenizer.eos_token
qwen_batch = qwen_tokenizer(["你好", "南京大学是一所大学"], padding=True, return_tensors="pt")
with torch.no_grad():
    batch_vectors = input_embedding(qwen_batch["input_ids"])
print("input_ids：", qwen_batch["input_ids"])
print("attention_mask：", qwen_batch["attention_mask"])
print("embedding shape：", tuple(batch_vectors.shape))
assert batch_vectors.shape[:2] == qwen_batch["attention_mask"].shape
print("相同 ID 的输入向量相同；padding 与 mask 已展示")


input_ids： tensor([[108386, 151643, 151643, 151643, 151643],
        [102034,  99562,  99639,  31838,  99562]])
attention_mask： tensor([[1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1]])
embedding shape： (2, 5, 1024)
相同 ID 的输入向量相同；padding 与 mask 已展示


### 回到小词表实验
下一步用小词表验证 one-hot 等价性，避免为真实模型构造巨大的 one-hot。
RoPE 在 attention 的 Q/K 上应用，不是直接加在这里的输入向量上。


In [7]:
del qwen, input_embedding
import gc
gc.collect()


## 实验 3：one-hot 等价性
用同一张表比较。实际实现不构建稠密 one-hot。

In [8]:
H = F.one_hot(I, num_classes=4).to(T.dtype)
torch.testing.assert_close(H @ T, E)
print("查表与 one-hot 矩阵乘法一致")


查表与 one-hot 矩阵乘法一致


## 附录：Tensor 操作
以下保留基础操作示例；与主线实验独立。

In [10]:
# 补充：Broadcasting（广播）机制详解

print("=== Broadcasting 基础概念 ===")
import torch

# 1. 标量与tensor的广播
print("--- 标量与tensor ---")
a = torch.tensor([1, 2, 3, 4])
b = 2
c = a + b
print(f"a: {a}")
print(f"b: {b}")
print(f"a + b: {c}")
print(f"形状: a={a.shape}, b=标量, c={c.shape}")


=== Broadcasting 基础概念 ===
--- 标量与tensor ---
a: tensor([1, 2, 3, 4])
b: 2
a + b: tensor([3, 4, 5, 6])
形状: a=torch.Size([4]), b=标量, c=torch.Size([4])


In [11]:

# 2. 不同形状tensor的广播
print("\n--- 不同形状tensor的广播 ---")
A = torch.randn(3, 4)
B = torch.randn(4)
print(f"A形状: {A.shape}")
print(f"B形状: {B.shape}")

print("A:", A)
print("B:", B)
# 广播过程：B [4] -> [1, 4] -> [3, 4]
C = A + B
print(f"A + B形状: {C.shape}")
print("C:", C)
print(f"广播是否成功: {C.shape == (3, 4)}")



--- 不同形状tensor的广播 ---
A形状: torch.Size([3, 4])
B形状: torch.Size([4])
A: tensor([[ 0.3367,  0.1288,  0.2345,  0.2303],
        [-1.1229, -0.1863,  2.2082, -0.6380],
        [ 0.4617,  0.2674,  0.5349,  0.8094]])
B: tensor([ 1.1103, -1.6898, -0.9890,  0.9580])
A + B形状: torch.Size([3, 4])
C: tensor([[ 1.4470, -1.5610, -0.7545,  1.1883],
        [-0.0126, -1.8761,  1.2192,  0.3200],
        [ 1.5719, -1.4224, -0.4541,  1.7673]])
广播是否成功: True


In [12]:
# Broadcasting结果一致性验证

print("=== 使用allclose验证Broadcasting结果一致性 ===")

# 1. 标量广播验证
print("--- 1. 标量广播验证 ---")
a = torch.tensor([1.0, 2.0, 3.0, 4.0])
b = 2.0

# 方法1：使用broadcasting
result_broadcast = a + b

# 方法2：手动编程
result_manual = a + torch.full_like(a, b)

print(torch.full_like(a,b))

print(f"原始tensor: {a}")
print(f"标量: {b}")
print(f"Broadcasting结果: {result_broadcast}")
print(f"手动编程结果: {result_manual}")
print(f"结果是否一致: {torch.allclose(result_broadcast, result_manual)}")
print(f"最大差异: {torch.max(torch.abs(result_broadcast - result_manual)):.10f}")


=== 使用allclose验证Broadcasting结果一致性 ===
--- 1. 标量广播验证 ---
tensor([2., 2., 2., 2.])
原始tensor: tensor([1., 2., 3., 4.])
标量: 2.0
Broadcasting结果: tensor([3., 4., 5., 6.])
手动编程结果: tensor([3., 4., 5., 6.])
结果是否一致: True
最大差异: 0.0000000000


In [13]:

# 2. 向量广播验证
print("\n--- 2. 向量广播验证 ---")
A = torch.randn(3, 4)
B = torch.randn(4)

# 方法1：使用broadcasting
result_broadcast = A + B

# 方法2：手动扩展
B_expanded = B.unsqueeze(0).expand(3, 4)

print("B:",B)
print("B_expanded:", B_expanded)
print("shape of B_expanded:", B_expanded.shape)

result_manual = A + B_expanded

print(f"矩阵A形状: {A.shape}")
print(f"向量B形状: {B.shape}")
print(f"Broadcasting结果形状: {result_broadcast.shape}")
print(f"手动编程结果形状: {result_manual.shape}")
print(f"结果是否一致: {torch.allclose(result_broadcast, result_manual)}")
print(f"最大差异: {torch.max(torch.abs(result_broadcast - result_manual)):.10f}")



--- 2. 向量广播验证 ---
B: tensor([-0.0499,  0.5263, -0.0085,  0.7291])
B_expanded: tensor([[-0.0499,  0.5263, -0.0085,  0.7291],
        [-0.0499,  0.5263, -0.0085,  0.7291],
        [-0.0499,  0.5263, -0.0085,  0.7291]])
shape of B_expanded: torch.Size([3, 4])
矩阵A形状: torch.Size([3, 4])
向量B形状: torch.Size([4])
Broadcasting结果形状: torch.Size([3, 4])
手动编程结果形状: torch.Size([3, 4])
结果是否一致: True
最大差异: 0.0000000000


In [14]:

# 3. 复杂广播验证
print("\n--- 3. 复杂广播验证 ---")
A = torch.randn(2, 3, 4)
B = torch.randn(3, 1)

# 方法1：使用broadcasting
result_broadcast = A + B

# 方法2：手动扩展
B_expanded = B.unsqueeze(0).expand(2, 3, 4)
result_manual = A + B_expanded

print("B:",B)
print("B_expanded:", B_expanded)
print("shape of B_expanded:", B_expanded.shape)


print(f"3D tensor A形状: {A.shape}")
print(f"2D tensor B形状: {B.shape}")
print(f"Broadcasting结果形状: {result_broadcast.shape}")
print(f"手动编程结果形状: {result_manual.shape}")
print(f"结果是否一致: {torch.allclose(result_broadcast, result_manual)}")
print(f"最大差异: {torch.max(torch.abs(result_broadcast - result_manual)):.10f}")



--- 3. 复杂广播验证 ---
B: tensor([[ 1.1351],
        [ 0.7592],
        [-3.5945]])
B_expanded: tensor([[[ 1.1351,  1.1351,  1.1351,  1.1351],
         [ 0.7592,  0.7592,  0.7592,  0.7592],
         [-3.5945, -3.5945, -3.5945, -3.5945]],

        [[ 1.1351,  1.1351,  1.1351,  1.1351],
         [ 0.7592,  0.7592,  0.7592,  0.7592],
         [-3.5945, -3.5945, -3.5945, -3.5945]]])
shape of B_expanded: torch.Size([2, 3, 4])
3D tensor A形状: torch.Size([2, 3, 4])
2D tensor B形状: torch.Size([3, 1])
Broadcasting结果形状: torch.Size([2, 3, 4])
手动编程结果形状: torch.Size([2, 3, 4])
结果是否一致: True
最大差异: 0.0000000000


In [15]:

# 4. 矩阵乘法中的广播验证
print("\n--- 4. 矩阵乘法中的广播验证 ---")
A = torch.randn(3, 4)
B = torch.randn(2, 4, 5)

# 方法1：使用matmul的广播
result_broadcast = torch.matmul(A, B)

# 方法2：手动扩展A然后计算
A_expanded = A.unsqueeze(0).expand(2, 3, 4)
result_manual = torch.bmm(A_expanded, B)


print("A:",A)
print("A_expanded:", A_expanded)
print("shape of A_expanded:", A_expanded.shape)

print(f"矩阵A形状: {A.shape}")
print(f"批量矩阵B形状: {B.shape}")
print(f"Broadcasting matmul结果形状: {result_broadcast.shape}")
print(f"手动bmm结果形状: {result_manual.shape}")
print(f"结果是否一致: {torch.allclose(result_broadcast, result_manual)}")
print(f"最大差异: {torch.max(torch.abs(result_broadcast - result_manual)):.10f}")



--- 4. 矩阵乘法中的广播验证 ---
A: tensor([[ 0.0192,  0.1052,  0.9603, -0.5672],
        [-0.5706,  1.5980,  0.1115, -0.0392],
        [ 1.4112, -0.6556,  0.8576, -1.6270]])
A_expanded: tensor([[[ 0.0192,  0.1052,  0.9603, -0.5672],
         [-0.5706,  1.5980,  0.1115, -0.0392],
         [ 1.4112, -0.6556,  0.8576, -1.6270]],

        [[ 0.0192,  0.1052,  0.9603, -0.5672],
         [-0.5706,  1.5980,  0.1115, -0.0392],
         [ 1.4112, -0.6556,  0.8576, -1.6270]]])
shape of A_expanded: torch.Size([2, 3, 4])
矩阵A形状: torch.Size([3, 4])
批量矩阵B形状: torch.Size([2, 4, 5])
Broadcasting matmul结果形状: torch.Size([2, 3, 5])
手动bmm结果形状: torch.Size([2, 3, 5])
结果是否一致: True
最大差异: 0.0000000000


In [16]:

# 5. 批量归一化中的广播验证
print("\n--- 5. 批量归一化中的广播验证 ---")
x = torch.randn(32, 64, 28, 28)

# 方法1：使用keepdim=True的广播
mean_broadcast = torch.mean(x, dim=(0, 2, 3), keepdim=True)
std_broadcast = torch.std(x, dim=(0, 2, 3), keepdim=True)
normalized_broadcast = (x - mean_broadcast) / (std_broadcast + 1e-8)

# 方法2：手动扩展
mean_manual = torch.mean(x, dim=(0, 2, 3))  # [64]
std_manual = torch.std(x, dim=(0, 2, 3))    # [64]
mean_expanded = mean_manual.view(1, 64, 1, 1).expand(32, 64, 28, 28)
std_expanded = std_manual.view(1, 64, 1, 1).expand(32, 64, 28, 28)
normalized_manual = (x - mean_expanded) / (std_expanded + 1e-8)

print(f"输入形状: {x.shape}")
print(f"Broadcasting方法结果形状: {normalized_broadcast.shape}")
print(f"手动扩展方法结果形状: {normalized_manual.shape}")
print(f"结果是否一致: {torch.allclose(normalized_broadcast, normalized_manual)}")
print(f"最大差异: {torch.max(torch.abs(normalized_broadcast - normalized_manual)):.10f}")



--- 5. 批量归一化中的广播验证 ---
输入形状: torch.Size([32, 64, 28, 28])
Broadcasting方法结果形状: torch.Size([32, 64, 28, 28])
手动扩展方法结果形状: torch.Size([32, 64, 28, 28])
结果是否一致: True
最大差异: 0.0000000000


In [17]:

# 6. 注意力机制中的广播验证
print("\n--- 6. 注意力机制中的广播验证 ---")
scores = torch.randn(2, 8, 8)
mask = torch.tensor([1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0])

# 方法1：使用unsqueeze的广播
masked_scores_broadcast = scores.masked_fill(~mask.bool().view(1, 1, -1), float("-inf"))

# 方法2：手动扩展
mask_expanded = mask.view(1, 1, 8).expand(2, 8, 8)
masked_scores_manual = scores.masked_fill(~mask_expanded.bool(), float("-inf"))

print(f"注意力分数形状: {scores.shape}")
print(f"mask形状: {mask.shape}")
print(f"Broadcasting方法结果形状: {masked_scores_broadcast.shape}")
print(f"手动扩展方法结果形状: {masked_scores_manual.shape}")
print(f"结果是否一致: {torch.allclose(masked_scores_broadcast, masked_scores_manual)}")
print(f"最大差异: {torch.max(torch.abs(masked_scores_broadcast[torch.isfinite(masked_scores_broadcast)] - masked_scores_manual[torch.isfinite(masked_scores_manual)])):.10f}")

print("\n=== 总结 ===")
print("✓ 所有broadcasting运算结果都与手动编程结果完全一致")
print("✓ 最大差异都在机器精度范围内（~1e-10）")
print("✓ 这证明了broadcasting机制的正确性和可靠性")



--- 6. 注意力机制中的广播验证 ---
注意力分数形状: torch.Size([2, 8, 8])
mask形状: torch.Size([8])
Broadcasting方法结果形状: torch.Size([2, 8, 8])
手动扩展方法结果形状: torch.Size([2, 8, 8])
结果是否一致: True
最大差异: 0.0000000000

=== 总结 ===
✓ 所有broadcasting运算结果都与手动编程结果完全一致
✓ 最大差异都在机器精度范围内（~1e-10）
✓ 这证明了broadcasting机制的正确性和可靠性


In [18]:

# 4. 广播失败的情况
print("\n--- 广播失败示例 ---")
try:
    A = torch.randn(3, 4)
    B = torch.randn(5)  # 不兼容的维度
    C = A + B
except RuntimeError as e:
    print(f"广播失败: {e}")

print("\n--- 广播规则总结 ---")
print("1. 从右向左对齐维度")
print("2. 维度必须兼容：相等、其中一个为1、或其中一个不存在")
print("3. 大小为 1 的兼容维度会自动扩展")



--- 广播失败示例 ---
广播失败: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 1

--- 广播规则总结 ---
1. 从右向左对齐维度
2. 维度必须兼容：相等、其中一个为1、或其中一个不存在
3. 大小为 1 的兼容维度会自动扩展


In [19]:
# 1. 形状变换操作：view vs reshape

print("=== view() vs reshape() 对比 ===")
import torch

# 创建示例tensor
x = torch.randn(2, 3, 4)
print(f"原始tensor形状: {x.shape}")
print(f"原始tensor是否连续: {x.is_contiguous()}")


=== view() vs reshape() 对比 ===
原始tensor形状: torch.Size([2, 3, 4])
原始tensor是否连续: True


In [20]:

# view() 操作
y1 = x.view(6, 4)  # 2*3=6
print(f"view(6, 4)后形状: {y1.shape}")
print(f"view后是否连续: {y1.is_contiguous()}")

# reshape() 操作
y2 = x.reshape(6, 4)
print(f"reshape(6, 4)后形状: {y2.shape}")
print(f"reshape后是否连续: {y2.is_contiguous()}")



view(6, 4)后形状: torch.Size([6, 4])
view后是否连续: True
reshape(6, 4)后形状: torch.Size([6, 4])
reshape后是否连续: True


In [21]:
print("\n=== 不连续tensor的处理 ===")
# 转置操作会使tensor不连续
x_transposed = x.transpose(0, 1)
print(f"转置后是否连续: {x_transposed.is_contiguous()}")

# view() 在不连续tensor上会报错
try:
    y3 = x_transposed.view(12, 2)
except RuntimeError as e:
    print(f"view()报错: {e}")


=== 不连续tensor的处理 ===
转置后是否连续: False
view()报错: view size is not compatible with input tensor's size and stride (at least one dimension spans across two contiguous subspaces). Use .reshape(...) instead.


In [22]:



# reshape() 会自动处理连续性问题
y4 = x_transposed.reshape(12, 2)
print(f"reshape()成功: {y4.shape}")

print("\n=== 内存共享验证 ===")
# 验证view()是否共享内存
x[0, 0, 0] = 999
print(f"修改原始tensor后，view结果: {y1[0, 0]}")
print(f"修改原始tensor后，reshape结果: {y2[0, 0]}")


reshape()成功: torch.Size([12, 2])

=== 内存共享验证 ===
修改原始tensor后，view结果: 999.0
修改原始tensor后，reshape结果: 999.0


In [23]:
# 2. 转置操作：transpose vs permute

print("=== transpose() - 交换两个维度 ===")
x = torch.randn(2, 3, 4, 5)
print(f"原始tensor形状: {x.shape}")

# transpose() 交换维度1和3
y1 = x.transpose(1, 3)
print(f"transpose(1, 3)后形状: {y1.shape}")

# 验证转置效果
print(f"原始tensor[0, 1, 2, 3] = {x[0, 1, 2, 3]}")
print(f"转置后tensor[0, 3, 2, 1] = {y1[0, 3, 2, 1]}")
print(f"两者是否相等: {x[0, 1, 2, 3] == y1[0, 3, 2, 1]}")


=== transpose() - 交换两个维度 ===
原始tensor形状: torch.Size([2, 3, 4, 5])
transpose(1, 3)后形状: torch.Size([2, 5, 4, 3])
原始tensor[0, 1, 2, 3] = 1.9732592105865479
转置后tensor[0, 3, 2, 1] = 1.9732592105865479
两者是否相等: True


In [24]:

print("\n=== permute() - 重新排列所有维度 ===")
# permute() 重新排列维度
y2 = x.permute(0, 3, 1, 2)
print(f"permute(0, 3, 1, 2)后形状: {y2.shape}")

# 验证permute效果
print(f"原始tensor[0, 1, 2, 3] = {x[0, 1, 2, 3]}")
print(f"permute后tensor[0, 3, 1, 2] = {y2[0, 3, 1, 2]}")
print(f"两者是否相等: {x[0, 1, 2, 3] == y2[0, 3, 1, 2]}")

print("\n=== 连续性问题 ===")
print(f"原始tensor是否连续: {x.is_contiguous()}")
print(f"transpose后是否连续: {y1.is_contiguous()}")
print(f"permute后是否连续: {y2.is_contiguous()}")

# 如果需要连续tensor，可以调用contiguous()
y1_cont = y1.contiguous()
print(f"contiguous()后是否连续: {y1_cont.is_contiguous()}")



=== permute() - 重新排列所有维度 ===
permute(0, 3, 1, 2)后形状: torch.Size([2, 5, 3, 4])
原始tensor[0, 1, 2, 3] = 1.9732592105865479
permute后tensor[0, 3, 1, 2] = 1.9732592105865479
两者是否相等: True

=== 连续性问题 ===
原始tensor是否连续: True
transpose后是否连续: False
permute后是否连续: False
contiguous()后是否连续: True


In [25]:
# 3. 维度操作：squeeze vs unsqueeze

print("=== squeeze() - 移除大小为1的维度 ===")
x = torch.randn(1, 3, 1, 4)
print(f"原始tensor形状: {x.shape}")

# squeeze() 移除所有大小为1的维度
y1 = x.squeeze()
print(f"squeeze()后形状: {y1.shape}")

# squeeze(dim) 只移除指定维度
y2 = x.squeeze(0)  # 只移除第0维
print(f"squeeze(0)后形状: {y2.shape}")

y3 = x.squeeze(2)  # 只移除第2维
print(f"squeeze(2)后形状: {y3.shape}")


=== squeeze() - 移除大小为1的维度 ===
原始tensor形状: torch.Size([1, 3, 1, 4])
squeeze()后形状: torch.Size([3, 4])
squeeze(0)后形状: torch.Size([3, 1, 4])
squeeze(2)后形状: torch.Size([1, 3, 4])


In [26]:

print("\n=== unsqueeze() - 在指定位置插入大小为1的维度 ===")
x = torch.randn(3, 4)
print(f"原始tensor形状: {x.shape}")

# unsqueeze() 在指定位置插入维度
y1 = x.unsqueeze(0)  # 在第0维插入
print(f"unsqueeze(0)后形状: {y1.shape}")

y2 = x.unsqueeze(-1)  # 在最后一维插入
print(f"unsqueeze(-1)后形状: {y2.shape}")

y3 = x.unsqueeze(1)  # 在第1维插入
print(f"unsqueeze(1)后形状: {y3.shape}")



=== unsqueeze() - 在指定位置插入大小为1的维度 ===
原始tensor形状: torch.Size([3, 4])
unsqueeze(0)后形状: torch.Size([1, 3, 4])
unsqueeze(-1)后形状: torch.Size([3, 4, 1])
unsqueeze(1)后形状: torch.Size([3, 1, 4])


In [27]:
# 6. 索引和选择操作

print("=== gather() - 按索引收集元素 ===")
# 创建示例tensor
x = torch.randn(3, 4)
print(f"原始tensor:\n{x}")

# 创建索引
indices = torch.tensor([0, 2, 1])
print(f"索引: {indices}")


=== gather() - 按索引收集元素 ===
原始tensor:
tensor([[ 0.3728,  1.4071, -2.3630,  0.4060],
        [ 0.4041, -1.0977,  0.6349, -0.7874],
        [-1.7962,  0.0679,  0.6599,  0.0105]])
索引: tensor([0, 2, 1])


In [28]:

# 在第1维上按索引收集元素
y = torch.gather(x, 1, indices.unsqueeze(1))
print(f"gather结果:\n{y}")
print(f"解释: 每行按indices选择元素")

# 验证结果
print(f"第0行选择第{indices[0]}列: {x[0, indices[0]]}")
print(f"第1行选择第{indices[1]}列: {x[1, indices[1]]}")
print(f"第2行选择第{indices[2]}列: {x[2, indices[2]]}")


gather结果:
tensor([[0.3728],
        [0.6349],
        [0.0679]])
解释: 每行按indices选择元素
第0行选择第0列: 0.3728405237197876
第1行选择第2列: 0.6349419951438904
第2行选择第1列: 0.06788241863250732


In [29]:

print("\n=== scatter() - 按索引分散元素 ===")
# 创建目标tensor
x_scatter = torch.zeros(3, 4)
values = torch.randn(3, 2)
indices_scatter = torch.tensor([[0, 2], [1, 3], [0, 1]])
print(f"目标tensor:\n{x_scatter}")
print(f"要分散的值:\n{values}")
print(f"分散索引:\n{indices_scatter}")



=== scatter() - 按索引分散元素 ===
目标tensor:
tensor([[0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])
要分散的值:
tensor([[-1.5884,  0.8354],
        [ 0.1429, -0.9776],
        [-0.6811,  1.2995]])
分散索引:
tensor([[0, 2],
        [1, 3],
        [0, 1]])


In [30]:

# 分散元素
x_scatter.scatter_(1, indices_scatter, values)
print(f"scatter后结果:\n{x_scatter}")

print("\n=== index_select() - 按索引选择 ===")
# 按索引选择行
x = torch.randn(5, 3)
indices = torch.tensor([0, 2, 4])
selected = torch.index_select(x, 0, indices)
print(f"原始tensor形状: {x.shape}")
print(f"选择的索引: {indices}")
print(f"选择结果形状: {selected.shape}")
print(f"选择结果:\n{selected}")


scatter后结果:
tensor([[-1.5884,  0.0000,  0.8354,  0.0000],
        [ 0.0000,  0.1429,  0.0000, -0.9776],
        [-0.6811,  1.2995,  0.0000,  0.0000]])

=== index_select() - 按索引选择 ===
原始tensor形状: torch.Size([5, 3])
选择的索引: tensor([0, 2, 4])
选择结果形状: torch.Size([3, 3])
选择结果:
tensor([[-0.8071, -0.8936, -0.5487],
        [ 0.5197, -0.6092, -0.3523],
        [-1.0002, -1.7554,  0.1185]])


In [31]:

print("\n=== masked_select() - 按掩码选择 ===")
# 创建掩码
x = torch.randn(3, 4)
mask = x > 0.5
print(f"原始tensor:\n{x}")
print(f"掩码 (>0.5):\n{mask}")

# 按掩码选择
selected_values = torch.masked_select(x, mask)
print(f"按掩码选择的值: {selected_values}")
print(f"选择的值数量: {len(selected_values)}")



=== masked_select() - 按掩码选择 ===
原始tensor:
tensor([[ 0.6266, -0.2781, -0.5870,  1.3485],
        [-1.5491,  1.2624,  1.5081, -0.3104],
        [-0.2782, -1.5248,  1.8441,  1.1037]])
掩码 (>0.5):
tensor([[ True, False, False,  True],
        [False,  True,  True, False],
        [False, False,  True,  True]])
按掩码选择的值: tensor([0.6266, 1.3485, 1.2624, 1.5081, 1.8441, 1.1037])
选择的值数量: 6
